Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

In [2]:
data_path = "C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Spring 2025\\STAT390\\LegalAid\\Data\\CAR_-_EP_Flow_Activity_Queue__Agent_Names\\"
adhoc_data_path = 'C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Adhoc\\Adhoc datasets\\'

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [4]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [5]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [6]:
df_main.shape

(3328626, 8)

In [7]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [8]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [9]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [10]:
df_main.reset_index(inplace = True)

In [11]:
familymenu_rows = df_main.loc[df_main['EP Name'] == 'Legal Family Menu Telephony EP',:]
familymenu_data = df_main.loc[df_main['Contact Session ID'].isin(familymenu_rows['Contact Session ID']),:]

In [14]:
familymenu_data.to_csv(adhoc_data_path + 'familymenudata.csv')

In [15]:
familymenu_data['family_menu_tag'] = 0
familymenu_data.loc[familymenu_data['Activity Name'] == 'FamilyMenu', 'family_menu_tag'] = 1

In [17]:
familymenutagcount = familymenu_data.groupby('Contact Session ID')['family_menu_tag'].sum()

In [19]:
familymenutagcount.value_counts()

family_menu_tag
1    22544
2     1951
3      242
4       48
5        5
0        3
7        2
6        2
Name: count, dtype: int64

In [20]:
familymenu_data['Contact Session ID'].nunique()

24797

In [34]:
familymenu_activity = familymenu_data[['Contact Session ID', 'Activity Name']]

In [36]:
familymenu_activity.dropna(subset=["Activity Name"], inplace=True)

In [45]:
familymenu_activity.groupby('Contact Session ID').first()

,Activity Name
Contact Session ID,
00014a58-a6ce-4cb2-a529-d55e2c9c304d,LanguageSelectionMenu
0004f7a4-ea7f-4d4e-9b63-33839fe9653d,LanguageSelectionMenu
0005118f-8ded-40d8-b52c-94752f25de68,LanguageSelectionMenu
000518df-2027-4c62-a31f-b9d35a76dbc4,LanguageSelectionMenu
0006a469-26a9-4687-a613-3dd2306760c0,LanguageSelectionMenu
...,...
fff898b5-9df2-40f4-a421-2dac3dc04194,LanguageSelectionMenu
fffabb74-7b6c-4f49-bea5-63001798762e,LanguageSelectionMenu
fffc8571-cff6-408f-ba73-65435f9cf9d4,LanguageSelectionMenu


In [56]:
df2 = familymenu_activity.copy()
df2["Next Activity"] = df2.groupby("Contact Session ID")["Activity Name"].shift(-1)

out = (
    df2.loc[df2["Activity Name"].eq("FamilyMenu"),
            ["Contact Session ID", "Activity Name", "Next Activity"]]
        .rename(columns={"Activity Name": "Current Activity"})
        .reset_index(drop=True)
)
out

,Contact Session ID,Current Activity,Next Activity
0,00014a58-a6ce-4cb2-a529-d55e2c9c304d,FamilyMenu,ClosedQueueMenu
1,0004f7a4-ea7f-4d4e-9b63-33839fe9653d,FamilyMenu,ClosedQueueMenu
2,0005118f-8ded-40d8-b52c-94752f25de68,FamilyMenu,DivorceOrParentingMenu
3,000518df-2027-4c62-a31f-b9d35a76dbc4,FamilyMenu,ClosedQueueMenu
4,0006a469-26a9-4687-a613-3dd2306760c0,FamilyMenu,ClosedQueueMenu
...,...,...,...
27410,fffabb74-7b6c-4f49-bea5-63001798762e,FamilyMenu,ClosedQueueMenu
27411,fffc8571-cff6-408f-ba73-65435f9cf9d4,FamilyMenu,LegalMenu2
27412,fffc8571-cff6-408f-ba73-65435f9cf9d4,FamilyMenu,DivorceOrParentingMenu
27413,fffceb0a-9302-48a6-be5b-67c542ccc539,FamilyMenu,ClosedQueueMenu


In [62]:
# Next activity and the one after that (per session), given rows are already sorted
df2 = familymenu_activity.copy()

g = df2.groupby("Contact Session ID")
df2["Second Activity"]      = g["Activity Name"].shift(-1)
df2["Third Activity"]  = g["Activity Name"].shift(-2)

out = (
    df2.loc[df2["Activity Name"].eq("FamilyMenu"),
            ["Contact Session ID", "Activity Name", "Second Activity", "Third Activity"]]
       .rename(columns={"Activity Name": "Current Activity"})
       .reset_index(drop=True)
)
out

,Contact Session ID,Current Activity,Second Activity,Third Activity
0,00014a58-a6ce-4cb2-a529-d55e2c9c304d,FamilyMenu,ClosedQueueMenu,NaN
1,0004f7a4-ea7f-4d4e-9b63-33839fe9653d,FamilyMenu,ClosedQueueMenu,ClosedQueueMenu
2,0005118f-8ded-40d8-b52c-94752f25de68,FamilyMenu,DivorceOrParentingMenu,ClosedQueueMenu
3,000518df-2027-4c62-a31f-b9d35a76dbc4,FamilyMenu,ClosedQueueMenu,ClosedQueueMenu
4,0006a469-26a9-4687-a613-3dd2306760c0,FamilyMenu,ClosedQueueMenu,NaN
...,...,...,...,...
27410,fffabb74-7b6c-4f49-bea5-63001798762e,FamilyMenu,ClosedQueueMenu,ClosedQueueMenu
27411,fffc8571-cff6-408f-ba73-65435f9cf9d4,FamilyMenu,LegalMenu2,FamilyMenu
27412,fffc8571-cff6-408f-ba73-65435f9cf9d4,FamilyMenu,DivorceOrParentingMenu,ClosedQueueMenu
27413,fffceb0a-9302-48a6-be5b-67c542ccc539,FamilyMenu,ClosedQueueMenu,NaN


In [73]:
out['Contact Session ID'].nunique()

24794

In [76]:
out.to_csv(adhoc_data_path + 'family_dash.csv', index = False)

In [77]:
out['Second Activity'].isna().sum()

543

In [80]:
custdata('0072c693-e22a-4376-bdab-3ad5ac92e066')

,index,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
5909,2248748,0072c693-e22a-4376-bdab-3ad5ac92e066,Main Number Telephony EP,NaN,NaN,2025-04-25 10:14:58,NaN,NaN,NaN,10
5910,2248749,0072c693-e22a-4376-bdab-3ad5ac92e066,NaN,LACMain,NaN,2025-04-25 10:14:58,NaN,NaN,NaN,10
5911,2248750,0072c693-e22a-4376-bdab-3ad5ac92e066,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-04-25 10:14:58,NaN,NaN,NaN,10
5912,2248751,0072c693-e22a-4376-bdab-3ad5ac92e066,Main Number Telephony EP,LACMain,NaN,2025-04-25 10:14:58,NaN,NaN,NaN,10
5913,2248757,0072c693-e22a-4376-bdab-3ad5ac92e066,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-04-25 10:15:16,NaN,NaN,NaN,10
5914,2248759,0072c693-e22a-4376-bdab-3ad5ac92e066,Main Number Telephony EP,NaN,MainMenu,2025-04-25 10:15:27,NaN,NaN,NaN,10
5915,2248764,0072c693-e22a-4376-bdab-3ad5ac92e066,NaN,PreLegalMenuSeniorsMenu,NaN,2025-04-25 10:15:51,NaN,NaN,NaN,10
5916,2248765,0072c693-e22a-4376-bdab-3ad5ac92e066,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-04-25 10:15:51,NaN,NaN,NaN,10
5917,2248766,0072c693-e22a-4376-bdab-3ad5ac92e066,NaN,LegalMenu,NaN,2025-04-25 10:16:00,NaN,NaN,NaN,10
5918,2248767,0072c693-e22a-4376-bdab-3ad5ac92e066,Legal Menu Telephony EP,NaN,LegalMenu1,2025-04-25 10:16:00,NaN,NaN,NaN,10


In [79]:
out.loc[(out['Second Activity'] == 'ClosedQueueMenu') & (out['Third Activity'] == 'ClinicVoicemailTransfer'),:]

,Contact Session ID,Current Activity,Second Activity,Third Activity
48,0072c693-e22a-4376-bdab-3ad5ac92e066,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
50,00750e54-15c8-400e-9212-da2652fbe7e2,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
72,00a38960-4ddc-46a3-a3f4-4c61dbea5b56,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
90,00c38e93-3d80-4d3b-95aa-b8c16eb4575a,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
113,0103ab8d-6e17-4184-85bf-7158f0ec7a23,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
...,...,...,...,...
27331,ff64bebf-b942-4198-82c5-399a63c4f901,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
27334,ff695c5a-6b85-497d-8839-50b0ddee647d,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
27372,ffbffc14-13c0-41ac-a1c3-210e5af643dc,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer
27376,ffc97bc9-1960-42ec-b368-216660677b40,FamilyMenu,ClosedQueueMenu,ClinicVoicemailTransfer


In [70]:
out.loc[out['Contact Session ID'] == '00930ed6-a66d-41c6-8a5c-f8be15d48af5',:]

,Contact Session ID,Current Activity,Second Activity,Third Activity
65,00930ed6-a66d-41c6-8a5c-f8be15d48af5,FamilyMenu,DivorceOrParentingMenu,ChildSupportMenu
66,00930ed6-a66d-41c6-8a5c-f8be15d48af5,FamilyMenu,NaN,NaN


In [69]:
custdata('00930ed6-a66d-41c6-8a5c-f8be15d48af5')

,index,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
7778,1789308,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Main Number Telephony EP,NaN,NaN,2024-11-19 08:16:52,NaN,NaN,NaN,8
7779,1789309,00930ed6-a66d-41c6-8a5c-f8be15d48af5,NaN,LACMain,NaN,2024-11-19 08:16:52,NaN,NaN,NaN,8
7780,1789310,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Main Number Telephony EP,NaN,LanguageSelectionMenu,2024-11-19 08:16:52,NaN,NaN,NaN,8
7781,1789311,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Main Number Telephony EP,LACMain,NaN,2024-11-19 08:16:52,NaN,NaN,NaN,8
7782,1789317,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Main Number Telephony EP,NaN,MainMenu,2024-11-19 08:17:01,NaN,NaN,NaN,8
7783,1789319,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Main Number Telephony EP,NaN,AppointmentMenu,2024-11-19 08:17:11,NaN,NaN,NaN,8
7784,1789325,00930ed6-a66d-41c6-8a5c-f8be15d48af5,NaN,PreLegalMenuSeniorsMenu,NaN,2024-11-19 08:17:19,NaN,NaN,NaN,8
7785,1789326,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2024-11-19 08:17:19,NaN,NaN,NaN,8
7786,1789328,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsConfirmationMenu,2024-11-19 08:17:27,NaN,NaN,NaN,8
7787,1789331,00930ed6-a66d-41c6-8a5c-f8be15d48af5,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SuburbsOrCityMenu,2024-11-19 08:17:46,NaN,NaN,NaN,8


In [68]:
out.loc[out['Second Activity'].isna(),:]

,Contact Session ID,Current Activity,Second Activity,Third Activity
66,00930ed6-a66d-41c6-8a5c-f8be15d48af5,FamilyMenu,NaN,NaN
173,01a24d58-e81c-4cd2-b82d-58424b189615,FamilyMenu,NaN,NaN
215,022c4500-4e63-4782-9b84-615b412b5642,FamilyMenu,NaN,NaN
371,03bc94bf-1a0a-4d3f-af08-587255e7592b,FamilyMenu,NaN,NaN
456,0476b112-a345-460f-881b-18e9e23f6477,FamilyMenu,NaN,NaN
...,...,...,...,...
27073,fcad8c93-7366-4a0e-9d47-fddd5177f3e6,FamilyMenu,NaN,NaN
27183,fdc2eabe-99b7-4b33-a752-188bf5614484,FamilyMenu,NaN,NaN
27290,fef977cc-0f55-49a7-b79a-bffdc4c0360f,FamilyMenu,NaN,NaN
27344,ff790916-286c-48e5-94fe-717f2af15581,FamilyMenu,NaN,NaN


In [71]:
out['Third Activity'].value_counts()

Third Activity
ClosedQueueMenu                     11818
SimpleDivorceMenu                    2394
ChildSupportMenu                     2289
FamilyMenu                           1859
IntakePreQueueMessage1               1449
ClinicVoicemailTransfer              1091
GetLoggedInFamilyAgents               914
LegalMenu2                            439
OtherLegalMenu                        320
HousingMenu                           226
BenefitsMenu                           86
GetLoggedInFamilySPAgents              76
HIVMenu                                67
GetLoggedInADAPTAgents                 27
CriminalRecordsVoicemailTransfer       24
EmploymentMenu                         20
ImmigrationMenu                        17
FrontDeskTransfer                      13
ClosedMenu                              8
GetLoggedInConsumerAgents               8
GetLoggedInADAPTSPAgents                3
DisconnectContact1                      1
AddressFaxHoursMenu                     1
Name: count, dtype:

In [59]:
out['Next Activity'].value_counts()

Next Activity
DivorceOrParentingMenu          12330
ClosedQueueMenu                  9414
LegalMenu2                       2882
GetLoggedInFamilyAgents          1499
TransferToSafeHaven               412
GetLoggedInFamilySPAgents         202
GetLoggedInEducationAgents        108
GetLoggedInEducationSPAgents       21
ClosedMenu                          4
Name: count, dtype: int64

In [52]:
second_idx = familymenu_activity.groupby('Contact Session ID').nth(3).index
second_rows = familymenu_activity.loc[second_idx]
second_rows['Activity Name'].value_counts()

Activity Name
LegalMenu1                        15548
SeniorsMenu                        5476
SeniorsConfirmationMenu            1893
LegalMenu2                          641
FamilyMenu                          578
MainMenu                            328
HelpWithLegalorOtherReasonMenu      114
AppointmentMenu                      93
SeniorsADAPTMenu                     55
SuburbsOrCityMenu                    25
OtherLegalMenu                       19
HIVMenu                               8
SeniorNotCookCoMenu                   5
AddressFaxHoursMenu                   5
HousingMenu                           4
BenefitsMenu                          2
ComplimentOrComplaintMenu             1
ImmigrationMenu                       1
EmploymentMenu                        1
Name: count, dtype: int64

In [43]:
custdata('fffceb0a-9302-48a6-be5b-67c542ccc539')

,index,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
3328562,608492,fffceb0a-9302-48a6-be5b-67c542ccc539,Main Number Telephony EP,NaN,NaN,2024-05-08 08:49:47,NaN,NaN,NaN,8
3328563,608493,fffceb0a-9302-48a6-be5b-67c542ccc539,NaN,LACMain,NaN,2024-05-08 08:49:47,NaN,NaN,NaN,8
3328564,608494,fffceb0a-9302-48a6-be5b-67c542ccc539,NaN,PreLegalMenuSeniorsMenu,NaN,2024-05-08 08:49:47,NaN,NaN,NaN,8
3328565,608495,fffceb0a-9302-48a6-be5b-67c542ccc539,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2024-05-08 08:49:47,NaN,NaN,NaN,8
3328566,608496,fffceb0a-9302-48a6-be5b-67c542ccc539,Pre-Legal Menu Seniors Menu Telephony EP,PreLegalMenuSeniorsMenu,NaN,2024-05-08 08:49:47,NaN,NaN,NaN,8
3328567,608506,fffceb0a-9302-48a6-be5b-67c542ccc539,NaN,LegalMenu,NaN,2024-05-08 08:50:00,NaN,NaN,NaN,8
3328568,608507,fffceb0a-9302-48a6-be5b-67c542ccc539,Legal Menu Telephony EP,NaN,LegalMenu1,2024-05-08 08:50:00,NaN,NaN,NaN,8
3328569,608513,fffceb0a-9302-48a6-be5b-67c542ccc539,Legal Menu Telephony EP,NaN,LegalMenu2,2024-05-08 08:50:50,NaN,NaN,NaN,8
3328570,608515,fffceb0a-9302-48a6-be5b-67c542ccc539,NaN,LegalFamilyMenu,NaN,2024-05-08 08:51:05,NaN,NaN,NaN,8
3328571,608516,fffceb0a-9302-48a6-be5b-67c542ccc539,Legal Family Menu Telephony EP,NaN,FamilyMenu,2024-05-08 08:51:05,NaN,NaN,NaN,8


In [31]:
familymenutagcount[familymenutagcount == 5]

Contact Session ID
5112ea60-2e73-4603-baee-bc01970dd180    5
a8434277-a571-4ff8-b4be-708ef42a2e6b    5
e91d1474-2298-4909-a76a-36481bc5f466    5
ec14e8c0-527a-45c5-af72-f1916af152c2    5
eeef251c-23df-4642-ae22-3821b340a5ed    5
Name: family_menu_tag, dtype: int64

In [33]:
df_main.loc[df_main['Contact Session ID'] == '5112ea60-2e73-4603-baee-bc01970dd180',:]

,index,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
1060165,146011,5112ea60-2e73-4603-baee-bc01970dd180,Main Number Telephony EP,NaN,NaN,2025-01-31 15:58:02,NaN,NaN,NaN,15
1060166,146012,5112ea60-2e73-4603-baee-bc01970dd180,NaN,LACMain,NaN,2025-01-31 15:58:02,NaN,NaN,NaN,15
1060167,146013,5112ea60-2e73-4603-baee-bc01970dd180,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-31 15:58:02,NaN,NaN,NaN,15
1060168,146014,5112ea60-2e73-4603-baee-bc01970dd180,Main Number Telephony EP,LACMain,NaN,2025-01-31 15:58:03,NaN,NaN,NaN,15
1060169,146015,5112ea60-2e73-4603-baee-bc01970dd180,Main Number Telephony EP,NaN,MainMenu,2025-01-31 15:58:14,NaN,NaN,NaN,15
1060170,146030,5112ea60-2e73-4603-baee-bc01970dd180,Main Number Telephony EP,NaN,MainMenu,2025-01-31 15:59:14,NaN,NaN,NaN,15
1060171,146031,5112ea60-2e73-4603-baee-bc01970dd180,NaN,PreLegalMenuSeniorsMenu,NaN,2025-01-31 15:59:17,NaN,NaN,NaN,15
1060172,146032,5112ea60-2e73-4603-baee-bc01970dd180,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-01-31 15:59:17,NaN,NaN,NaN,15
1060173,146034,5112ea60-2e73-4603-baee-bc01970dd180,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-01-31 15:59:28,NaN,NaN,NaN,15
1060174,146038,5112ea60-2e73-4603-baee-bc01970dd180,NaN,LegalMenu,NaN,2025-01-31 15:59:38,NaN,NaN,NaN,15
